In [ ]:
# Install required packages (run once)
%pip install -q -r ../../requirements.txt

# Module 8: Production Deployment

In this final module, we bring everything together for production deployment.

## What You'll Learn

1. Production memory system architecture
2. Graceful degradation and fallbacks
3. Observability and monitoring
4. Deployment patterns

## Production Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        Load Balancer                            │
└─────────────────────────────────┬───────────────────────────────┘
                                  │
          ┌───────────────────────┼───────────────────────┐
          │                       │                       │
          ▼                       ▼                       ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Agent Pod 1   │     │   Agent Pod 2   │     │   Agent Pod 3   │
│  MemorySystem   │     │  MemorySystem   │     │  MemorySystem   │
└────────┬────────┘     └────────┬────────┘     └────────┬────────┘
         │                       │                       │
         └───────────────────────┼───────────────────────┘
                                 │
    ┌───────────┬────────────────┼────────────────┬───────────┐
    │           │                │                │           │
    ▼           ▼                ▼                ▼           ▼
┌───────┐  ┌────────┐     ┌──────────┐     ┌──────────┐  ┌───────┐
│Cosmos │  │ Neo4j  │     │   Redis  │     │AI Search │  │Foundry│
│  DB   │  │        │     │  Cache   │     │          │  │       │
└───────┘  └────────┘     └──────────┘     └──────────┘  └───────┘
 Episodic   Semantic       Shared         Procedural      LLM
```

In [ ]:
# Setup
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import nest_asyncio
nest_asyncio.apply()

from dotenv import load_dotenv
load_dotenv('../.env')

print("✓ Environment loaded")

## Production Memory System

The `MemorySystem` class provides:

- Unified initialization of all memory types
- Graceful fallback to in-memory stores
- Health checks for monitoring
- Observability and metrics

---
## Graceful Degradation

Production systems must handle service failures without complete outage.

### Circuit Breaker + Fallback Pattern

```python
async def query(self, query: str):
    if self.primary_available:
        try:
            return await self.primary.query(query)
        except ConnectionError:
            self.primary_available = False
    return await self.fallback.query(query)  # In-memory
```

### Degradation Matrix

| Memory Type | Primary | Fallback | Behavior When Degraded |
|-------------|---------|----------|----------------------|
| Episodic | Cosmos DB | In-memory | Session-only history |
| Semantic | Neo4j | In-memory graph | Limited relations |
| Procedural | AI Search | Sample data | Basic policies |
| Shared | Redis | Local store | No cross-instance sync |

---

## Health Check Types

Kubernetes and container orchestrators need three probe types:

| Probe | Question | Response |
|-------|----------|----------|
| **Liveness** | Is the process running? | 200 OK if alive |
| **Readiness** | Can it accept traffic? | 200 OK if ready to serve |
| **Startup** | Has initialization completed? | 200 OK when initialized |

---

## Observability (Three Pillars)

### 1. Metrics

| Metric | Type | Description |
|--------|------|-------------|
| `memory.tool.calls` | Counter | Tool invocations by name |
| `memory.tool.errors` | Counter | Failed tool calls |
| `memory.tool.latency_ms` | Histogram | Response time distribution |
| `memory.fallback.active` | Gauge | Number of active fallbacks |

### 2. Structured Logging

```python
logger.info("memory.query", memory_type="episodic", user_id=user_id, query_length=len(query))
```

### 3. Distributed Tracing

```python
with tracer.start_as_current_span("memory.query") as span:
    span.set_attribute("memory.type", "episodic")
    result = await memory.query(request)
```

In [ ]:
from production_memory import (
    MemoryConfig,
    MemorySystem,
    PRODUCTION_AGENT_INSTRUCTIONS,
    create_production_system
)

# Create configuration from environment
config = MemoryConfig.from_env()

print("Configuration:")
print(f"  Foundry Endpoint: {config.foundry_endpoint[:50]}...")
print(f"  Model: {config.model_deployment}")
print(f"  Cosmos DB: {'Configured' if config.cosmos_endpoint else 'Fallback'}")
print(f"  Neo4j: {'Configured' if config.neo4j_uri else 'Fallback'}")
print(f"  AI Search: {'Configured' if config.search_endpoint else 'Fallback'}")
print(f"  Redis: {'Configured' if config.redis_url else 'Fallback'}")

In [ ]:
# Initialize the production system
system = MemorySystem(config)
await system.initialize()

print(f"\n✓ Memory system initialized with {len(system.tools)} tools")

## Health Checks

Production systems need health endpoints:

In [ ]:
import json

health = await system.health_check()

print("Health Status:")
print(json.dumps(health, indent=2))

## Create Production Agent

Create an agent with all memory tools:

In [ ]:
# Create the production agent
agent = system.create_agent(
    name="ProductionTravelAssistant",
    instructions=PRODUCTION_AGENT_INSTRUCTIONS
)

print(f"✓ Production agent created with {len(system.tools)} memory tools")

In [ ]:
async def production_chat(message: str):
    """Chat with the production agent."""
    print(f"👤 {message}\n")
    
    async with agent:
        stream = await agent.run(message, stream=True)
        response = await stream.get_final_response()
        print(f"🤖 {response}")
        print("\n" + "="*60 + "\n")

# Test the production agent
await production_chat(
    "I'm planning a trip to London for a conference. "
    "What are my preferences and what approvals do I need?"
)

## Graceful Degradation

The system continues working even when services are unavailable:

| Service Down | Fallback | Impact |
|-------------|----------|--------|
| Cosmos DB | In-memory episodic | No cross-session history |
| Neo4j | In-memory semantic | No persistent preferences |
| AI Search | In-memory procedural | Limited policy set |
| Redis | In-memory shared | No cross-instance sync |

In [ ]:
# Simulate degraded mode
print("Degradation Status:")
for service, status in health["services"].items():
    if status == "in-memory":
        print(f"  ⚠️ {service}: Using fallback (in-memory)")
    elif status == "error":
        print(f"  ❌ {service}: Error (unavailable)")
    else:
        print(f"  ✅ {service}: Production ({status})")

## Observability

Track metrics for monitoring:

In [ ]:
from production_memory import MemoryMetrics
import time

metrics = MemoryMetrics()

# Simulate some tool calls
tools_used = [
    ("recall_events_local", 45.2, True),
    ("query_user_knowledge_local", 32.1, True),
    ("find_procedure_local", 28.5, True),
    ("recall_events_local", 48.7, True),
    ("store_shared_local", 15.3, True),
    ("recall_events_local", 150.0, False),  # Simulated error
]

for tool_name, latency, success in tools_used:
    metrics.record_call(tool_name, latency, success)

print("Metrics Summary:")
print(json.dumps(metrics.get_summary(), indent=2))

## Deployment Options

### Option 1: Azure Container Apps

```yaml
# azure-container-app.yaml
properties:
  template:
    containers:
    - name: travel-agent
      image: myregistry.azurecr.io/travel-agent:latest
      env:
      - name: FOUNDRY_PROJECT_ENDPOINT
        secretRef: foundry-endpoint
      - name: COSMOS_ENDPOINT
        secretRef: cosmos-endpoint
      - name: NEO4J_URI
        secretRef: neo4j-uri
      - name: AZURE_SEARCH_ENDPOINT
        secretRef: search-endpoint
      - name: REDIS_URL
        secretRef: redis-url
```

### Option 2: Azure Kubernetes Service

```yaml
# deployment.yaml
apiVersion: apps/v1
kind: Deployment
spec:
  replicas: 3
  template:
    spec:
      containers:
      - name: travel-agent
        image: myregistry.azurecr.io/travel-agent:latest
        envFrom:
        - secretRef:
            name: memory-secrets
        livenessProbe:
          httpGet:
            path: /health
            port: 8000
```

## Dockerfile Example

In [ ]:
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application
COPY . .

# Health check endpoint
HEALTHCHECK --interval=30s --timeout=10s \\
  CMD curl -f http://localhost:8000/health || exit 1

# Run
EXPOSE 8000
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print(dockerfile)

## Environment Variables Summary

| Variable | Description | Required |
|----------|-------------|----------|
| `FOUNDRY_PROJECT_ENDPOINT` | Azure AI Foundry endpoint | Yes |
| `FOUNDRY_MODEL_DEPLOYMENT_NAME` | Model deployment name | Yes |
| `COSMOS_ENDPOINT` | Cosmos DB endpoint | No (fallback) |
| `NEO4J_URI` | Neo4j connection URI | No (fallback) |
| `AZURE_SEARCH_ENDPOINT` | AI Search endpoint | No (fallback) |
| `REDIS_URL` | Redis connection URL | No (fallback) |

## Production Checklist

### Security
- [ ] Use managed identities (no keys in code)
- [ ] Enable Entra ID authentication
- [ ] Configure network isolation (VNets)
- [ ] Enable audit logging

### Reliability
- [ ] Configure health checks
- [ ] Set up auto-scaling
- [ ] Implement retry logic
- [ ] Test fallback scenarios

### Observability
- [ ] Enable Application Insights
- [ ] Configure log analytics
- [ ] Set up alerts on error rates
- [ ] Track latency metrics

### Performance
- [ ] Tune connection pool sizes
- [ ] Configure caching (Redis)
- [ ] Optimize query patterns
- [ ] Test under load

## Summary

Congratulations! You've completed the Agentic Memory Tutorial.

## What You've Learned

| Module | Topic | Key Concept |
|--------|-------|-------------|
| 1 | Agent Basics | Microsoft Agent Framework + Session Memory |
| 2 | Episodic Memory | Events with Cosmos DB |
| 3 | Semantic Memory | Knowledge graphs with Neo4j |
| 4 | Procedural Memory | Policies with AI Search |
| 5 | Combined Memory | Multi-memory synthesis |
| 6 | Memory Handoff | Multi-agent workflows |
| 7 | Shared Memory | Distributed state with Redis |
| 8 | Production | Deployment patterns |

## Memory Architecture

```
                        ┌─────────────────┐
                        │   User Query    │
                        └────────┬────────┘
                                 │
                        ┌────────▼────────┐
                        │  Memory Router  │
                        └────────┬────────┘
                                 │
         ┌───────────────────────┼───────────────────────┐
         │                       │                       │
         ▼                       ▼                       ▼
   ┌───────────┐           ┌───────────┐           ┌───────────┐
   │ Episodic  │           │ Semantic  │           │Procedural │
   │ Cosmos DB │           │   Neo4j   │           │ AI Search │
   │           │           │           │           │           │
   │"What      │           │"What do   │           │"How do    │
   │happened?" │           │I know?"   │           │I do it?"  │
   └───────────┘           └───────────┘           └───────────┘
         │                       │                       │
         └───────────────────────┼───────────────────────┘
                                 │
                        ┌────────▼────────┐
                        │   Synthesize    │
                        │    Response     │
                        └─────────────────┘
```

## Next Steps

1. **Deploy** your agent using the patterns in this module
2. **Customize** memory schemas for your domain
3. **Extend** with additional memory types
4. **Monitor** and optimize in production

## Resources

- [Microsoft Agent Framework Documentation](https://learn.microsoft.com/azure/ai-services/agents/)
- [Azure AI Foundry](https://ai.azure.com/)
- [Azure Cosmos DB](https://docs.microsoft.com/azure/cosmos-db/)
- [Neo4j AuraDB](https://neo4j.com/cloud/aura/)
- [Azure AI Search](https://docs.microsoft.com/azure/search/)
- [Azure Cache for Redis](https://docs.microsoft.com/azure/azure-cache-for-redis/)